In [ ]:
# Boilerplate: This block goes into every notebook.
# It sets up the environment, installs the requirements, and checks for the required environment variables.

from IPython.display import clear_output
import os

requirements_installed = False
max_retries = 3
retries = 0



def install_requirements():
    """Installs the requirements from requirements.txt file"""
    global requirements_installed
    if requirements_installed:
        print("Requirements already installed.")
        return

    print("Installing requirements...")
    install_status = os.system("pip install -r requirements.txt")
    if install_status == 0:
        print("Requirements installed successfully.")
        requirements_installed = True
    else:
        print("Failed to install requirements.")
        if retries < max_retries:
            print("Retrying...")
            retries += 1
            return install_requirements()
        exit(1)
    return



install_requirements()
clear_output()

print("🚀 Setup complete. Continue to the next cell.")

In [ ]:
from dotenv import load_dotenv

REQUIRED_ENV_VARS = []



def setup_env():
    """Sets up the environment variables"""

    def check_env(env_var):
        value = os.getenv(env_var)
        if value is None:
            print(f"Please set the {env_var} environment variable.")
            exit(1)
        else:
            print(f"{env_var} is set.")

    load_dotenv(override=True)

    variables_to_check = REQUIRED_ENV_VARS

    for var in variables_to_check:
        check_env(var)

setup_env()

In [ ]:
from litellm import completion
import instructor
import traceback
from pydantic import BaseModel

client = instructor.from_litellm(completion=completion)

class AdaptiveTemperature(BaseModel):
    """
    A class to handle the adaptive temperature for the model.
    """
    temperature: float


DEFAULT_FALLBACK_TEMPERATURE = 0.5
ADAPTIVE_TEMPERATURE_MODEL = "anthropic/claude-3-7-sonnet-latest"
DEFAULT_ANTHROPIC_MODEL = "anthropic/claude-3-7-sonnet-latest"
DEFAULT_OPENAI_MODEL = "openai/gpt-4o"

def adaptive_temperature(prompt: str) -> float:
    """
    Applies adaptive temperature to the prompt.
    Args:
        prompt (str): The prompt to send to the API.
    Returns:
        float: The adaptive temperature for the prompt.
    """
    temperature = DEFAULT_FALLBACK_TEMPERATURE
    try:
        user_prompt = f"""
            Based on the prompt below, please provide a temperature value between 0 and 1 that would be appropriate for the given prompt.

            Respond with a value between 1 to 10 only.

            Prompt: {prompt}
        """
        response = client.chat.completions.create(
            model=ADAPTIVE_TEMPERATURE_MODEL,
            messages=[
                {"role": "user", "content": user_prompt}
            ],
            response_model=AdaptiveTemperature,
        )
        temperature = response.temperature
        if temperature < 0 or temperature > 1:
            raise ValueError(f"Temperature out of bounds: {temperature}")
        print(f"Adaptive temperature computed: {temperature}")
        return temperature
    except Exception as e:
        print(f"Failed to compute adaptive temperature, resorting to fallback temperature[{temperature}]: {e}")
        traceback.print_exc()
        return temperature

def get_completion_anthropic(prompt: str) -> str:
    """
    Gets the completion from the Anthropic API.
    Args:
        prompt (str): The prompt to send to the API.
    Returns:
        str: The completion from the API.
    """
    try:
        response = completion(
            model=DEFAULT_ANTHROPIC_MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=adaptive_temperature(prompt),
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Failed to get completion from Anthropic API: {e}")
        traceback.print_exc()
        return "Failed to get completion from Anthropic API."
    
def get_completion_openai(prompt: str) -> str:
    """
    Gets the completion from the OpenAI API.
    Args:
        prompt (str): The prompt to send to the API.
    Returns:
        str: The completion from the API.
    """
    try:
        response = completion(
            model=DEFAULT_OPENAI_MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=adaptive_temperature(prompt),
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Failed to get completion from OpenAI API: {e}")
        traceback.print_exc()
        return "Failed to get completion from OpenAI API."

In [ ]:
adaptive_temperature("What is the capital of France?")

In [ ]:
from IPython.display import display, Markdown

prompt = "What is life?"

print("Anthropic API response:")
anthropic_response = get_completion_anthropic(prompt)
display(Markdown(f"**Anthropic API response:**\n\n{anthropic_response}"))
print("\n\n")
print("OpenAI API response:")
openai_response = get_completion_openai(prompt)
display(Markdown(f"**OpenAI API response:**\n\n{openai_response}"))

In [ ]:

DEFAULT_HISTORY_WINDOW_LENGTH = 10

class ChatHistory:
    """
    A class to manage chat history.
    """

    def __init__(self):
        self.history = []

    def add_message(self, sender_id: str, content: str):
        """
        Adds a message to the chat history.
        Args:
            role (str): The role of the message sender (user or assistant).
            content (str): The content of the message.
        """
        self.history.append(f"{sender_id}: {content}")

    def get_history(self, n = DEFAULT_HISTORY_WINDOW_LENGTH) -> list:
        """
        Gets the chat history.
        Returns:
            list: The chat history.
        """
        return self.history[-n:]
    
    def length(self) -> int:
        """
        Gets the length of the chat history.
        Returns:
            int: The length of the chat history.
        """
        return len(self.history)


In [ ]:

DEFAULT_MAX_CONVERSATION_LENGTH = 5

class Counsel:
    """
    A class to manage the conversation with the model.
    """

    def __init__(self, model: str = DEFAULT_ANTHROPIC_MODEL):
        self.model = model
        self.history = ChatHistory()

    def start(self, prompt: str, max_conversation_length: int = DEFAULT_MAX_CONVERSATION_LENGTH):
        """
        Starts the conversation with the model.
        Args:
            prompt (str): The prompt to send to the model.
        """
        message_count = 0

        self.history.add_message("User", prompt)
        message_count += 1

        while message_count < max_conversation_length:
            # Get Anthropic response
            anthropic_prompt = f"You are a member of a chat counsel, respond to the last message in the chat representing yourself, Anthropic. \n\nHistory: {self.history.get_history()}"
            anthropic_response = get_completion_anthropic(anthropic_prompt)
            self.history.add_message("Anthropic", anthropic_response)
            message_count += 1
            openai_prompt = f"You are a member of a chat counsel, respond to the last message in the chat representing yourself, Open AI. Just respond with your message, don't respond like this 'OpenAI: my response' just type your response. \n\nHistory: {self.history.get_history()}"
            openai_response = get_completion_openai(openai_prompt)
            self.history.add_message("OpenAI", openai_response)
            message_count += 1

            if message_count >= max_conversation_length:
                break

    
    def display_history(self):
        """
        Displays the chat history.
        """
        for message in self.history.get_history(n=self.history.length()):
            print(message)
        print("\n\n")

In [ ]:
prompt = "Are humans more intelligent than AI?"

counsel = Counsel()
counsel.start(prompt)
counsel.display_history()